# Radial synthesis — multi-scenario

Radial wheel of the planetary boundaries (LP): 1 sector per LP, 1 radial bar
per scenario (height = value / Safe Boundary, so the Safe Boundary is a circle of
constant radius = 1L), annotated outer ring (overshoot level for the reference
scenario, % variation for the others) and High-Risk Boundary arc specific to
each sector (the High-Risk Boundary / Safe Boundary ratio varies with the LP).
No color gradient between scenarios: only the angular position (and the text
of the outer ring) distinguishes them within a sector.

Reuses the same data as `footprint_decomposition.ipynb`
(`create_footprint_decomposition`), shown in a radial form rather than as a
grid of subplots.

Generates 2 figures in `figures/radial_synthesis/<SCENARIO_GROUP>/` (`create_radial_synthesis`),
each one with the Excel workbook of its data (same name, `.xlsx`):
- `radial_synthesis_cba_all_scenarios.png`: solid bar, colored by LP.
- `radial_synthesis_pba_all_scenarios.png`: solid bar, colored by LP --
  the PBA is a single scalar in the source data, with no breakdown by
  category or domestic/imported possible.

Computation and plotting logic shared with the other notebooks through the
[`planefr_lib`](./planefr_lib/) package.

In [ ]:
# ============================ PARAMETERS ============================
SCENARIO_GROUP = "CONF"           # France scenarios (e.g. "CONF", "ADEME", "no_SSP"...); "<group>_agri" = "Agri" sheets
SHARING_PRINCIPLE = "EPC"         # national budget = world budget x share in budget_shares.xlsx:
                                  #   "EPC" (equal per capita) or "CTR" (capability to reduce)
RADIAL_POWER = 0.4                # radius = (value / Safe Boundary) ** power: 1 = linear, 0.5 = sqrt, smaller = more dilated
SCALE_UB = True                   # True: scale rescaled per LP so that the High-Risk Boundary is at 2L; False: common nL scale
SCALE_DIFF = False                # True: radial axis capped at 6L, larger bars truncated (incompatible with SCALE_UB)
DISPLAY_BOUNDS = True             # True: Lower/High-Risk Boundary arcs shown in addition to the Safe Boundary circle
SHOW_DLS = True                   # (CBA) True: Decent Living Standards (social floor) arcs shown
BDF = None                        # (CBA) None: scenarios; or a sheet of the BdF file (e.g. "décile"): household categories
TITLE_CBA = None                  # figure titles (None = automatic)
TITLE_PBA = "Planetary boundaries assessment of French NZE Scenarios by 2050\n Production-based (territorial)"
DPI = 300                         # resolution of the saved .png
# ====================================================================

import sys
from pathlib import Path

notebooks_dir = Path.cwd()
while not (notebooks_dir / "planefr_lib").exists() and notebooks_dir != notebooks_dir.parent:
    notebooks_dir = notebooks_dir.parent
if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

import matplotlib.pyplot as plt

from planefr_lib import config, io, processing
from planefr_lib.plot_radial_synthesis import create_radial_synthesis

# Folder of the figures (and of their Excel workbooks): figures/<NOTEBOOK_NAME>/<SCENARIO_GROUP>/
NOTEBOOK_NAME = "radial_synthesis"

## 1. Loading the data and processing the scenarios

In [ ]:
cf_df = io.load_characterization_factors(SCENARIO_GROUP)
bridge_matrices_df = io.load_bridge_matrices()
boundaries_df = io.load_boundaries(SCENARIO_GROUP)
dls_df = io.load_dls()

group_figures_dir = config.figure_dir(NOTEBOOK_NAME, SCENARIO_GROUP)
scenario_folders = config.get_scenario_folders(exclude_2019=True, scenario_group=SCENARIO_GROUP)
scenario_names = [f.name for f in scenario_folders]
print(f"Scenarios found ({len(scenario_folders)}): {scenario_names}")

all_scenarios_data = []
for scenario_folder in scenario_folders:
    print(f"Processing: {scenario_folder.name}")
    data_by_indicator, _ = processing.process_scenario(scenario_folder, cf_df, bridge_matrices_df, boundaries_df)
    all_scenarios_data.append(data_by_indicator)
assert len(all_scenarios_data) == len(scenario_folders), "Some scenarios produced no data"
print(f"\n{len(all_scenarios_data)}/{len(scenario_folders)} scenarios processed successfully.")

## 2. Radial wheel — consumption-based accounting (CBA)

The bars are solid, colored by LP (domestic + imported merged), like the PBA figure below.

`radial_power=1/3` dilates the radial scale more strongly than a square root:
the LPs below their Safe Boundary (e.g. Blue water consumption) remain visible,
without the LPs in large overshoot (e.g. GHG emissions) squashing the rest of
the figure. Raise it towards 1.0 for a strictly linear scale.

In [ ]:
output_path = group_figures_dir / "radial_synthesis_cba_all_scenarios.png"
fig_cba, _ = create_radial_synthesis(
    all_scenarios_data, boundaries_df, scenario_names, show_cba=True, show_pba=False,
    dls_df=dls_df, show_dls=SHOW_DLS, Bdf=BDF, radial_power=RADIAL_POWER,
    display_bounds=DISPLAY_BOUNDS, scale_diff=SCALE_DIFF, scale_UB=SCALE_UB,
    title=TITLE_CBA, sharing_principle=SHARING_PRINCIPLE, scenario_group=SCENARIO_GROUP,
    excel_path=output_path.with_suffix(".xlsx"))
fig_cba.savefig(output_path, dpi=DPI, bbox_inches="tight")
print(f"Saved: {output_path}")
plt.show()

## 3. Radial wheel — production-based accounting (PBA)

The PBA is a single scalar in the source data (no breakdown by category or
domestic/imported): the bar is therefore always solid, colored by LP.

In [ ]:
output_path = group_figures_dir / "radial_synthesis_pba_all_scenarios.png"
fig_pba, _ = create_radial_synthesis(
    all_scenarios_data, boundaries_df, scenario_names, show_cba=False, show_pba=True,
    dls_df=dls_df, show_dls=False,  # the DLS is a consumption floor: not shown on the PBA wheel
    radial_power=RADIAL_POWER, display_bounds=DISPLAY_BOUNDS, scale_diff=SCALE_DIFF, scale_UB=SCALE_UB,
    title=TITLE_PBA, sharing_principle=SHARING_PRINCIPLE, scenario_group=SCENARIO_GROUP,
    excel_path=output_path.with_suffix(".xlsx"))
fig_pba.savefig(output_path, dpi=DPI, bbox_inches="tight")
print(f"Saved: {output_path}")
plt.show()